In [1]:
import os
from pathlib import Path

print("Current working directory:", os.getcwd())
print()
print("What's in the current directory:")
for item in Path(".").iterdir():
    print(" -", item.name)
print()
print("Does '../data/raw' exist from here?", (Path("..") / "data" / "raw").exists())
print("Does './data/raw' exist from here?", (Path(".") / "data" / "raw").exists())

Current working directory: c:\Users\kedha\Documents\dfw-hospital-pricing\notebooks

What's in the current directory:
 - 01_sanity_check.ipynb
 - 02_first_pull.ipynb
 - 03_anomaly_investigation.ipynb
 - 04_cross_hospital_commercial.ipynb
 - 05_cross_hospital_chart.ipynb
 - 06_v2_classifier.ipynb
 - 07_pricing_analysis_1.ipynb
 - 08_discounted_cash_price_2.ipynb
 - 09_cash_vs_lob_ladder_4.ipynb

Does '../data/raw' exist from here? True
Does './data/raw' exist from here? False


In [2]:
import duckdb
import pandas as pd
from pathlib import Path

DATA_DIR = Path("..") / "data" / "raw"

HOSPITAL_FILES = {
    "Baylor University Medical Center (Dallas)":
        "baylor_university_medical_center-69947_parsed.duckdb",
    "Methodist Dallas Medical Center":
        "methodist_dallas_medical_center-6000b_parsed.duckdb",
    "Parkland Health (Dallas)":
        "parkland_health-6e88d_parsed.duckdb",
    "Texas Health Presbyterian Hospital Plano":
        "texas_health_presbyterian_hospital_plano-6ad81_parsed.duckdb",
    "Medical City Alliance Hospital (Fort Worth)":
        "medical_city_alliance_hospital-77912_parsed.duckdb",
}
KNEE_MRI_CPT = "73721"
TARGET_PAYERS = ["Blue Cross", "UnitedHealthcare", "Aetna"]
for hospital, filename in HOSPITAL_FILES.items():
    path = DATA_DIR / filename
    status = "✓" if path.exists() else "✗ MISSING"
    print(f"{status} {hospital}: {filename}")

✓ Baylor University Medical Center (Dallas): baylor_university_medical_center-69947_parsed.duckdb
✓ Methodist Dallas Medical Center: methodist_dallas_medical_center-6000b_parsed.duckdb
✓ Parkland Health (Dallas): parkland_health-6e88d_parsed.duckdb
✓ Texas Health Presbyterian Hospital Plano: texas_health_presbyterian_hospital_plano-6ad81_parsed.duckdb
✓ Medical City Alliance Hospital (Fort Worth): medical_city_alliance_hospital-77912_parsed.duckdb


In [3]:
baylor_path = DATA_DIR / HOSPITAL_FILES["Baylor University Medical Center (Dallas)"]
print(f"Connecting to: {baylor_path}")
print(f"File size: {baylor_path.stat().st_size / 1024 / 1024:.1f} MB")
print()

con = duckdb.connect(str(baylor_path), read_only=True)

tables = con.execute("SHOW TABLES").fetchdf()
print("Tables in this database:")
print(tables)

Connecting to: ..\data\raw\baylor_university_medical_center-69947_parsed.duckdb
File size: 68.5 MB

Tables in this database:
                      name
0         cms_hpt_metadata
1                hospitals
2  modifier_charge_details
3         modifier_charges
4             mrf_metadata
5  standard_charge_details
6         standard_charges


In [4]:
schema = con.execute("DESCRIBE standard_charge_details").fetchdf()
print(f"Columns in standard_charge_details: {len(schema)} total")
print()
print(schema)

with pd.option_context('display.max_rows', None):
    print(schema)

Columns in standard_charge_details: 44 total

                   column_name column_type null   key default extra
0                    detail_id      BIGINT   NO  None    None  None
1                    charge_id      BIGINT  YES  None    None  None
2                   charge_seq     INTEGER  YES  None    None  None
3                    payer_seq     INTEGER  YES  None    None  None
4                  hospital_id      BIGINT  YES  None    None  None
5                  description     VARCHAR  YES  None    None  None
6                 gross_charge      DOUBLE  YES  None    None  None
7              discounted_cash      DOUBLE  YES  None    None  None
8                      minimum      DOUBLE  YES  None    None  None
9                      maximum      DOUBLE  YES  None    None  None
10                     setting     VARCHAR  YES  None    None  None
11               billing_class     VARCHAR  YES  None    None  None
12    additional_generic_notes     VARCHAR  YES  None    None  None
13

In [5]:
required_cols = [
    'cpt', 'description', 'gross_charge', 'setting',
    'payer_name', 'payer_group', 'payer_type', 'plan_name',
    'standard_charge_dollar', 'standard_charge_percentage'
]

actual_cols = set(schema['column_name'].tolist())

print("Column check:")
for col in required_cols:
    status = "✓" if col in actual_cols else "✗ MISSING"
    print(f"  {status} {col}")

Column check:
  ✓ cpt
  ✓ description
  ✓ gross_charge
  ✓ setting
  ✓ payer_name
  ✓ payer_group
  ✓ payer_type
  ✓ plan_name
  ✓ standard_charge_dollar
  ✓ standard_charge_percentage


In [6]:

query = f"""
    SELECT
        description,
        setting,
        payer_name,
        payer_group,
        payer_type,
        standard_charge_dollar,
        gross_charge
    FROM standard_charge_details
    WHERE cpt = '{KNEE_MRI_CPT}'
    LIMIT 10
"""
result = con.execute(query).fetchdf()
print(f"Found {len(result)} rows (capped at 10)")
result

Found 0 rows (capped at 10)


,description,setting,payer_name,payer_group,payer_type,standard_charge_dollar,gross_charge


In [7]:
q1 = "SELECT COUNT(*) AS total_rows, COUNT(cpt) AS rows_with_cpt FROM standard_charge_details"
print("--- Hypothesis 1: Is cpt populated? ---")
print(con.execute(q1).fetchdf())
print()

q2 = """
    SELECT cpt, COUNT(*) AS row_count
    FROM standard_charge_details
    WHERE cpt IN ('73721', '73722', '73723')
    GROUP BY cpt
"""
print("--- Hypothesis 2: Any knee MRI codes (73721/73722/73723)? ---")
print(con.execute(q2).fetchdf())
print()

q3 = """
    SELECT cpt, COUNT(*) AS row_count
    FROM standard_charge_details
    WHERE cpt LIKE '737%'
    GROUP BY cpt
    ORDER BY cpt
"""
print("--- Hypothesis 3: Any CPT codes starting with '737' (MRI lower extremity)? ---")
print(con.execute(q3).fetchdf())
print()

q4 = """
    SELECT DISTINCT description, cpt, hcpcs
    FROM standard_charge_details
    WHERE LOWER(description) LIKE '%knee%'
      AND (LOWER(description) LIKE '%mri%' OR LOWER(description) LIKE '%magnetic%')
    LIMIT 10
"""
print("--- Hypothesis 4: Descriptions containing 'knee' and 'MRI'/'magnetic'? ---")
print(con.execute(q4).fetchdf())

--- Hypothesis 1: Is cpt populated? ---
   total_rows  rows_with_cpt
0     1851086          67173

--- Hypothesis 2: Any knee MRI codes (73721/73722/73723)? ---
Empty DataFrame
Columns: [cpt, row_count]
Index: []

--- Hypothesis 3: Any CPT codes starting with '737' (MRI lower extremity)? ---
Empty DataFrame
Columns: [cpt, row_count]
Index: []

--- Hypothesis 4: Descriptions containing 'knee' and 'MRI'/'magnetic'? ---
                                         description    cpt hcpcs
0  INJECTION PROCEDURE FOR CONTRAST KNEE ARTHROGR...  27369  None


In [8]:


qB = """
    SELECT DISTINCT description, cpt, hcpcs, rc
    FROM standard_charge_details
    WHERE LOWER(description) LIKE '%mri%'
       OR LOWER(description) LIKE '%magnetic resonance%'
    LIMIT 20
"""
print("--- B: Any MRI descriptions (any body part) ---")
print(con.execute(qB).fetchdf())
print()

qC = """
    SELECT cpt, description, setting, COUNT(*) as row_count
    FROM standard_charge_details
    WHERE cpt IS NOT NULL
    GROUP BY cpt, description, setting
    ORDER BY row_count DESC
    LIMIT 10
"""
print("--- C: Top 10 most common CPT-coded services ---")
print(con.execute(qC).fetchdf())

--- B: Any MRI descriptions (any body part) ---
                                          description    cpt  hcpcs   rc
0                                   HC CAD BREAST MRI    NaN  C8937  610
1      HC INJ SHOULDER ARTHROGRAPHY CT/MRI ARTHG - LT    NaN  23350  361
2   HC BS INGEVITY MRI PACEMAKER LEAD (PASSIVE/ACT...    NaN  C1898  275
3                           HC MRI CHEST W/O CONTRAST    NaN  71550  610
4                         HC MRI ABDOMEN W/O CONTRAST    NaN  74181  610
5                        HC MRI EXT LOWER W/O&W CONTR    NaN  73720  610
6      HC INJ SHOULDER ARTHROGRAPHY CT/MRI ARTHG - RT    NaN  23350  361
7                     HC MRI BONE MARROW BLOOD SUPPLY    NaN  77084  610
8                      HC HIGH RISK ABR-MRI BUMC ONLY    NaN    NaN  999
9    HC MRI,CARDIAC - MORPHOLGY & FUNCTN W/O CONTRAST    NaN  75557  610
10      HC MEDTRONIC VISIA AF MRI VR SURESCAN DVFB1D4    NaN  C1722  275
11                       HC MRI SPINE LUMB W/CONTRAST    NaN  72149  612
12 

In [9]:
query = f"""
    SELECT
        description,
        setting,
        cpt,
        hcpcs,
        rc,
        payer_name,
        payer_group,
        payer_type,
        standard_charge_dollar,
        gross_charge
    FROM standard_charge_details
    WHERE cpt = '{KNEE_MRI_CPT}'
       OR hcpcs = '{KNEE_MRI_CPT}'
    LIMIT 20
"""
result = con.execute(query).fetchdf()
print(f"Found {len(result)} rows (capped at 20)")
result

Found 20 rows (capped at 20)


,description,setting,cpt,hcpcs,rc,payer_name,payer_group,payer_type,standard_charge_dollar,gross_charge
0,HC MR LWR EXT JT WO CM LT,outpatient,None,73721,610,Aetna,Aetna,Commercial,1875.12,3409.3
1,HC MR LWR EXT JT WO CM LT,outpatient,None,73721,610,Aetna,Aetna,Commercial,224.10,3409.3
2,HC MR LWR EXT JT WO CM LT,outpatient,None,73721,610,American Health Plan,Other,Other,235.30,3409.3
3,HC MR LWR EXT JT WO CM LT,outpatient,None,73721,610,Baylor Scott & White Health Plan,Other,Other,2056.72,3409.3
4,HC MR LWR EXT JT WO CM LT,outpatient,None,73721,610,Baylor Scott & White Health Plan,Other,Other,369.76,3409.3
5,HC MR LWR EXT JT WO CM LT,outpatient,None,73721,610,Baylor Scott & White Health Plan,Other,Other,281.25,3409.3
6,HC MR LWR EXT JT WO CM LT,outpatient,None,73721,610,Baylor Scott & White Health Plan,Other,Other,1748.21,3409.3
7,HC MR LWR EXT JT WO CM LT,outpatient,None,73721,610,Baylor Scott & White Health Plan,Other,Other,268.92,3409.3
8,HC MR LWR EXT JT WO CM LT,outpatient,None,73721,610,Baylor Scott & White Health Plan,Other,Other,212.90,3409.3
9,HC MR LWR EXT JT WO CM LT,outpatient,None,73721,610,Blue Cross Blue Shield,BCBS,Commercial,722.42,3409.3


In [10]:
query = f"""
    SELECT 
        payer_name,
        payer_group,
        payer_type,
        COUNT(*) AS row_count
    FROM standard_charge_details
    WHERE (cpt = '{KNEE_MRI_CPT}' OR hcpcs = '{KNEE_MRI_CPT}')
    GROUP BY payer_name, payer_group, payer_type
    ORDER BY row_count DESC
"""
result = con.execute(query).fetchdf()
print(f"Distinct payers offering knee MRI at Baylor: {len(result)}")
result

Distinct payers offering knee MRI at Baylor: 26


,payer_name,payer_group,payer_type,row_count
0,Baylor Scott & White Health Plan,Other,Other,24
1,Blue Cross Blue Shield,BCBS,Commercial,24
2,United Healthcare,UnitedHealthcare,Commercial,16
3,QuickTrip,Other,Other,8
4,Cigna,Cigna,Commercial,8
5,Employers Health Network,Other,Other,8
6,HealthSmart,Other,Other,8
7,WellPoint (fka Amerigroup),Other,Other,8
8,Texas Workforce Commission,Other,Other,8
9,Humana,Humana,Commercial,8


In [11]:
query = f"""
    SELECT
        payer_group,
        payer_name,
        plan_name,
        setting,
        standard_charge_dollar,
        standard_charge_percentage,
        gross_charge
    FROM standard_charge_details
    WHERE (cpt = '{KNEE_MRI_CPT}' OR hcpcs = '{KNEE_MRI_CPT}')
      AND payer_group IN ('BCBS', 'UnitedHealthcare', 'Aetna')
    ORDER BY payer_group, standard_charge_dollar
"""
result = con.execute(query).fetchdf()
print(f"Knee MRI rates at Baylor for BCBS / UHC / Aetna: {len(result)} rows")
result

Knee MRI rates at Baylor for BCBS / UHC / Aetna: 48 rows


,payer_group,payer_name,plan_name,setting,standard_charge_dollar,standard_charge_percentage,gross_charge
0,Aetna,Aetna,Medicare Advantage,outpatient,224.10,NaN,3409.3
1,Aetna,Aetna,Medicare Advantage,outpatient,224.10,NaN,3409.3
2,Aetna,Aetna,Medicare Advantage,outpatient,224.10,NaN,3409.3
3,Aetna,Aetna,Medicare Advantage,outpatient,224.10,NaN,3409.3
4,Aetna,Aetna,Commercial,outpatient,1875.12,NaN,3409.3
5,Aetna,Aetna,Commercial,outpatient,1875.12,NaN,3409.3
6,Aetna,Aetna,Commercial,outpatient,1875.12,NaN,3409.3
7,Aetna,Aetna,Commercial,outpatient,1875.12,NaN,3409.3
8,BCBS,Blue Cross Blue Shield,Medicare Advantage,outpatient,235.30,NaN,3409.3
9,BCBS,Blue Cross Blue Shield,Medicare Advantage,outpatient,235.30,NaN,3409.3


In [12]:
query = f"""
    WITH distinct_rates AS (
        SELECT DISTINCT
            payer_group,
            payer_name,
            plan_name,
            standard_charge_dollar
        FROM standard_charge_details
        WHERE (cpt = '{KNEE_MRI_CPT}' OR hcpcs = '{KNEE_MRI_CPT}')
          AND payer_group IN ('BCBS', 'UnitedHealthcare', 'Aetna')
          AND standard_charge_dollar IS NOT NULL
    )
    SELECT
        payer_group,
        COUNT(*) AS plan_count,
        MIN(standard_charge_dollar) AS min_rate,
        MEDIAN(standard_charge_dollar) AS median_rate,
        MAX(standard_charge_dollar) AS max_rate,
        ROUND(AVG(standard_charge_dollar), 2) AS avg_rate
    FROM distinct_rates
    GROUP BY payer_group
    ORDER BY payer_group
"""
result = con.execute(query).fetchdf()
print("Baylor knee MRI summary (distinct plan-level rates):")
result

Baylor knee MRI summary (distinct plan-level rates):


,payer_group,plan_count,min_rate,median_rate,max_rate,avg_rate
0,Aetna,2,224.10,1049.610,1875.12,1049.61
1,BCBS,6,235.30,1198.995,1526.41,1055.27
2,UnitedHealthcare,4,257.72,2358.000,2620.00,1898.43


In [13]:
con.close()

all_results = []

for hospital_name, filename in HOSPITAL_FILES.items():
    db_path = DATA_DIR / filename
    print(f"Querying: {hospital_name} ... ", end="")
    
    try:
        con = duckdb.connect(str(db_path), read_only=True)
        
        query = f"""
            WITH distinct_rates AS (
                SELECT DISTINCT
                    payer_group,
                    payer_name,
                    plan_name,
                    standard_charge_dollar
                FROM standard_charge_details
                WHERE (cpt = '{KNEE_MRI_CPT}' OR hcpcs = '{KNEE_MRI_CPT}')
                  AND payer_group IN ('BCBS', 'UnitedHealthcare', 'Aetna')
                  AND standard_charge_dollar IS NOT NULL
            )
            SELECT
                payer_group,
                COUNT(*) AS plan_count,
                MIN(standard_charge_dollar) AS min_rate,
                MEDIAN(standard_charge_dollar) AS median_rate,
                MAX(standard_charge_dollar) AS max_rate,
                ROUND(AVG(standard_charge_dollar), 2) AS avg_rate
            FROM distinct_rates
            GROUP BY payer_group
            ORDER BY payer_group
        """
        df = con.execute(query).fetchdf()
        df['hospital'] = hospital_name
        all_results.append(df)
        
        con.close()
        print(f"got {len(df)} payer groups")
    except Exception as e:
        print(f"ERROR: {e}")

combined = pd.concat(all_results, ignore_index=True)

combined = combined[['hospital', 'payer_group', 'plan_count', 
                     'min_rate', 'median_rate', 'max_rate', 'avg_rate']]
print(f"\nTotal rows across all hospitals: {len(combined)}")
combined

Querying: Baylor University Medical Center (Dallas) ... got 3 payer groups
Querying: Methodist Dallas Medical Center ... got 3 payer groups
Querying: Parkland Health (Dallas) ... got 3 payer groups
Querying: Texas Health Presbyterian Hospital Plano ... got 3 payer groups
Querying: Medical City Alliance Hospital (Fort Worth) ... got 2 payer groups

Total rows across all hospitals: 14


,hospital,payer_group,plan_count,min_rate,median_rate,max_rate,avg_rate
0,Baylor University Medical Center (Dallas),Aetna,2,224.10,1049.610,1875.12,1049.61
1,Baylor University Medical Center (Dallas),BCBS,6,235.30,1198.995,1526.41,1055.27
2,Baylor University Medical Center (Dallas),UnitedHealthcare,4,257.72,2358.000,2620.00,1898.43
3,Methodist Dallas Medical Center,Aetna,12,232.47,1600.000,3414.60,1428.17
4,Methodist Dallas Medical Center,BCBS,9,232.47,1381.930,1593.99,1128.30
5,Methodist Dallas Medical Center,UnitedHealthcare,7,200.30,232.470,232.47,227.87
6,Parkland Health (Dallas),Aetna,14,208.80,963.440,7278.60,1665.24
7,Parkland Health (Dallas),BCBS,23,678.20,4368.240,8734.32,5916.13
8,Parkland Health (Dallas),UnitedHealthcare,6,1274.07,3937.500,7873.00,4573.76
9,Texas Health Presbyterian Hospital Plano,Aetna,5,224.14,1053.000,1539.00,840.06


In [14]:
output_path = Path("..") / "data" / "processed" / "day3_knee_mri_summary.csv"
combined.to_csv(output_path, index=False)
print(f"Saved {len(combined)} rows to {output_path}")
print(f"File size: {output_path.stat().st_size} bytes")

Saved 14 rows to ..\data\processed\day3_knee_mri_summary.csv
File size: 1153 bytes
